# 🛡️ PharmaGuard — AI Pharmacy Security Monitor
### Cloud-Assisted Real-Time Suspicious-Interaction Monitoring System

---

## 1. Introduction

**PharmaGuard** is an AI-assisted real-time surveillance system designed for pharmacy operations. It monitors designated pharmacy shelf areas (e.g. prescription medicines or controlled substances), tracks individuals across frames, detects suspicious behavioral interaction sequences (such as prolonged dwell/loitering or rapid grab-and-retreat movement), and alerts pharmacy staff for human review.

### Cloud-Assisted Architecture
The system employs a browser-to-cloud computer vision pipeline:
```
[ Laptop Webcam ]
        │  (Browser WebRTC/WebSocket Video Capture)
        ▼
[ Secure HTTPS Live Tunnel ] (Gradio .live tunnel)
        │
        ▼
[ Google Colab GPU Cloud Runtime ]
        ├─► [ Person Detection ]  (Pretrained YOLOv8 / RT-DETR)
        ├─► [ Multi-Object MOT ]  (ByteTrack Two-Stage Association)
        ├─► [ Behavior Analysis ] (Shelf ROI Dwell Time & Loitering)
        ├─► [ Alert Management ]  (Severity Grading & Cooldown Throttling)
        └─► [ Frame Annotation ]  (Bounding Boxes, IDs, Trails & HUD)
        │
        ▼
[ Browser Monitoring Dashboard ] (Real-Time Video + Telemetry + Alerts)
```


---

## 2. Runtime Configuration

Inspect the active Colab environment, Python runtime version, CUDA drivers, and GPU hardware allocation. A GPU runtime (NVIDIA T4, L4, V100, or A100) is strongly recommended for real-time inference.


In [ ]:
import sys
import platform
import torch

print("=" * 65)
print("🔍 PHARMAGUARD RUNTIME ENVIRONMENT CHECK")
print("=" * 65)

# 1. Python Environment
print(f"Python Version:   {platform.python_version()} ({sys.executable})")

# 2. PyTorch & CUDA Availability
print(f"PyTorch Version:  {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:   {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_version = torch.version.cuda
    print(f"Active GPU Model: {gpu_name} ({total_mem_gb:.2f} GB VRAM)")
    print(f"Allocated GPUs:   {gpu_count}")
    print(f"CUDA Toolkit:     {cuda_version}")
    print("\n✅ GPU hardware detected. Real-time inference acceleration enabled.")
else:
    print("\n⚠️  WARNING: NO GPU DETECTED!")
    print("   Running on CPU will limit processing throughput to ~8-10 FPS.")
    print("   To switch to GPU: Click Runtime > Change runtime type > Select T4 GPU > Save.")
print("=" * 65)


---

## 3. Repository Setup

Clone the official PharmaGuard GitHub repository into `/content/pharmguard` and verify that all core modules exist. If the directory already exists, it pulls the latest updates from remote.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Franklinnn099/Pharmguard.git"
TARGET_DIR = "/content/pharmguard"

print("=" * 65)
print("📁 PHARMAGUARD REPOSITORY INITIALIZATION")
print("=" * 65)

if os.path.exists(TARGET_DIR):
    print(f"Repository directory already exists at {TARGET_DIR}.")
    print("Pulling latest commits from GitHub...")
    os.chdir(TARGET_DIR)
    res = subprocess.run(["git", "pull"], capture_output=True, text=True, check=False)
    print(res.stdout.strip() or res.stderr.strip())
elif os.path.exists("pharmaguard") and os.path.exists("main.py"):
    print(f"Already running from project directory: {os.getcwd()}")
else:
    print(f"Cloning repository from {REPO_URL} into {TARGET_DIR}...")
    res = subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], capture_output=True, text=True, check=False)
    if res.returncode == 0:
        print("Repository cloned successfully.")
    else:
        print(f"Clone error: {res.stderr}")
    os.chdir(TARGET_DIR)

# Verify presence of all required project files
required_files = [
    "main.py",
    "benchmark.py",
    "requirements.txt",
    "pharmaguard/__init__.py",
    "pharmaguard/config.py",
    "pharmaguard/detector.py",
    "pharmaguard/tracker.py",
    "pharmaguard/behavior.py",
    "pharmaguard/alerts.py",
    "pharmaguard/events.py",
    "pharmaguard/annotator.py",
    "pharmaguard/pipeline.py",
    "pharmaguard/ui.py",
    "pharmaguard/app.py",
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    print(f"❌ Warning: Missing required project files: {missing_files}")
else:
    print(f"✅ All {len(required_files)} core modules and runners verified in {os.getcwd()}.")
print("=" * 65)


---

## 4. Dependency Installation

Install the pinned dependencies from `requirements.txt`.
- Handles package installations cleanly without arbitrary version changes.
- Checks compatibility of Gradio and Pydantic schemas to ensure smooth WebSocket streaming.
- Verifies critical vision and machine learning imports.


In [ ]:
import sys
import subprocess

print("=" * 65)
print("📦 INSTALLING PHARMAGUARD DEPENDENCIES")
print("=" * 65)

# Install pinned dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user", "lap"], check=False)

# Verify critical package imports and versions
import torch
import cv2
import ultralytics
import gradio as gr
import pydantic
import numpy as np

print("\nPackage Verification:")
print(f"  - PyTorch:      {torch.__version__}")
print(f"  - OpenCV:       {cv2.__version__}")
print(f"  - Ultralytics:  {ultralytics.__version__}")
print(f"  - Gradio:       {gr.__version__}")
print(f"  - Pydantic:     {pydantic.__version__}")
print(f"  - NumPy:        {np.__version__}")

# Check Gradio / Pydantic JSON-schema compatibility
try:
    import jsonschema
    print(f"  - JSONSchema:   {jsonschema.__version__}")
    print("✅ Schema serialization verified clean.")
except ImportError:
    print("ℹ️ jsonschema module optional; default validation active.")

print("=" * 65)


---

## 5. Authentication Configuration

PharmaGuard requires authentication on public tunnels to prevent unauthorized viewing of the pharmacy surveillance feed.

### How Authentication Works:
1. **Google Colab Secrets (Recommended)**: Set `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Colab's left sidebar (🔑 Secrets icon) and toggle "Notebook access".
2. **Environment Variables**: Configure your desired username and password in the cell below.
3. **Automatic Ephemeral Token**: If no credentials are set, `pharmaguard/app.py` will generate a secure, random one-time password and display it upon launch.


In [ ]:
import os
import getpass

print("=" * 65)
print("🔐 CONFIGURE DASHBOARD AUTHENTICATION")
print("=" * 65)

# 1. Attempt retrieval from Google Colab Secrets
colab_user, colab_pass = None, None
try:
    from google.colab import userdata
    colab_user = userdata.get("PHARMAGUARD_AUTH_USER")
    colab_pass = userdata.get("PHARMAGUARD_AUTH_PASS")
except Exception:
    pass

if colab_user and colab_pass:
    os.environ["PHARMAGUARD_AUTH_USER"] = colab_user
    os.environ["PHARMAGUARD_AUTH_PASS"] = colab_pass
    print(f"✅ Loaded credentials securely from Colab Secrets (Operator: {colab_user})")
elif os.environ.get("PHARMAGUARD_AUTH_USER") and os.environ.get("PHARMAGUARD_AUTH_PASS"):
    print(f"✅ Loaded credentials from environment (Operator: {os.environ.get('PHARMAGUARD_AUTH_USER')})")
else:
    print("ℹ️  No static credentials configured.")
    print("   PharmaGuard will generate a one-time ephemeral session password on startup.")
    print("   (Optional: to configure your own credentials, set PHARMAGUARD_AUTH_USER and PHARMAGUARD_AUTH_PASS)")
print("=" * 65)


---

## 6. Model Initialization & Verification

Preload the pretrained model weights (`yolov8n.pt`), verify that the model is allocated to the GPU, and run an initial warmup inference on an empty test frame.


In [ ]:
import torch
import numpy as np
from pharmaguard.config import DetectorConfig
from pharmaguard.detector import YOLODetector

print("=" * 65)
print("🤖 INITIALIZING VISION DETECTOR")
print("=" * 65)

target_device = "cuda" if torch.cuda.is_available() else "cpu"
detector_cfg = DetectorConfig(model_name="yolov8n.pt", device=target_device)

print(f"Loading {detector_cfg.model_name} onto compute device: {target_device.upper()}...")
detector = YOLODetector(detector_cfg)

model_info = detector.get_model_info()
print("Model Metadata:")
print(f"  - Weights File:        {model_info.get('model_name')}")
print(f"  - Active Compute Unit: {model_info.get('device').upper()}")
print(f"  - Confidence Filter:   {model_info.get('confidence_threshold')}")
print(f"  - Target Classes:      {model_info.get('target_classes')} (Person)")

# Warmup inference
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
test_output = detector.detect(dummy_frame)
print(f"Warmup inference completed successfully (Detections: {test_output.count}).")
print("✅ Model loaded and ready for real-time tracking.")
print("=" * 65)


---

## 7. GPU Benchmark

Run the automated benchmark suite (`benchmark.py`) to measure processing latency across all 4 stages:
1. **Stage 1 (Detection)**: YOLO letterbox preprocessing, PyTorch forward pass, NMS, GPU tensor transfer.
2. **Stage 2 (Tracking)**: ByteTrack two-stage Hungarian association (`scipy.optimize.linear_sum_assignment`).
3. **Stage 3 (Behavior)**: Pharmacy shelf polygon ROI ground-contact testing and dwell tracking.
4. **Stage 4 (Annotation)**: Rendering shelf polygons, bounding boxes, motion trails, and Top HUD.

> ⚠️ **Important Note on Benchmark Metrics**:
> This benchmark evaluates **backend processing throughput** on 50 synthetic test frames at 640x480 resolution with hardware synchronization (`torch.cuda.synchronize()`).
> It measures raw computational capability on Colab GPU hardware. It is **not** a measurement of live webcam streaming FPS over the internet (which is additionally governed by browser capture rate, video compression, and network roundtrip).


In [ ]:
import sys
import subprocess

# Run full benchmark suite with CUDA synchronization guards
subprocess.run([sys.executable, "benchmark.py"], check=True)


---

## 8. Launch PharmaGuard Security Console

Start the live PharmaGuard surveillance server with public HTTPS tunneling enabled (`--colab`).

### Instructions to Connect Your Laptop Webcam:
1. Run the cell below.
2. Look for the public tunnel link printed in the output: **`Running on public URL: https://xxxx.gradio.live`**.
3. Click the link to open the dashboard in your laptop browser.
4. **Log in**:
   - If you configured credentials in Section 5, enter them.
   - Otherwise, copy the **`[SECURITY] Generated one-time ephemeral session credentials`** printed in the output below.
5. In the dashboard, click the **Webcam** input on the left panel.
6. When prompted by your browser (**Allow access to your camera?**), click **Allow**.
7. Live video frames will transmit from your browser to this Colab GPU, and real-time annotated video with bounding boxes, tracking IDs, and dwell alerts will return to your screen.


In [ ]:
import sys
import subprocess

# Launch PharmaGuard with Colab configuration
subprocess.run([sys.executable, "main.py", "--colab", "--model", "yolov8n.pt"])


---

## 9. Live System Verification Guide

Once your laptop webcam is streaming to the Colab GPU backend, perform the following verification steps:

| Test Scenario | Action | Expected Visual & Behavioral Result |
| :--- | :--- | :--- |
| **1. Person Detection** | Step in front of the laptop webcam. | Bounding box with `ID #1` appears around your body. Confidence score and label are visible. |
| **2. Multi-Person Tracking** | Have a second person enter the camera view. | A second bounding box appears with `ID #2`. IDs remain stable when crossing paths without switching. |
| **3. Shelf Zone Entry** | Move your feet into `Prescription_Shelf_A` (left side of feed). | Shelf polygon turns from Teal/Secured to Amber/Occupied. Label updates to `ID #1 \| X.Xs`. |
| **4. Prolonged Dwell Alert** | Stand inside the shelf zone for more than 8 seconds. | Top HUD highlights alert count; a red `[ALERT] PROLONGED DWELL` card appears in the alert feed. |
| **5. Rapid Retreat Alert** | Step into the shelf zone for ~1 second and quickly step out. | Amber `[WARNING] RAPID RETREAT` alert appears in the event timeline. |
| **6. Disconnect Recovery** | Click the camera stop button or switch browser tabs. | Dashboard status switches cleanly to `STANDBY (Webcam Paused/Inactive)` without crashing. Resume to continue tracking. |

> 💡 **Performance Note**:  
> Distinguish **backend GPU processing time** (typically ~8–12 ms per frame) from **browser-visible streaming rate** (~20–25 FPS). Real-time browser streaming is smooth because frame transmission runs over low-latency WebSockets with `stream_every=0.04`.


---

## 10. Troubleshooting & Common Questions

### 1. GPU Unavailable (`CUDA Available: False`)
- **Cause**: Google Colab started in CPU runtime mode.
- **Solution**: From the Colab top menu, select **Runtime > Change runtime type > Hardware accelerator > T4 GPU** and click **Save**. Re-run from Section 2.

### 2. Gradio JSON-Schema Warning or Error
- **Cause**: Mismatched `pydantic` or `jsonschema` library versions in Colab.
- **Solution**: Run `!pip install -q --upgrade "pydantic>=2.5.0" "jsonschema>=4.19.0"` and restart the session (**Runtime > Restart session**).

### 3. Webcam Permission Denied in Browser
- **Cause**: Browser security policy or conflicting application.
- **Solution**: Modern browsers strictly require **HTTPS** for camera access. Always open the public `https://xxxx.gradio.live` link (never an unencrypted `http://` link). Ensure no other application (Zoom, Teams, Skype) is currently using your webcam.

### 4. Public Gradio Live Tunnel Disconnected
- **Cause**: Network timeout or Colab cellular/Wi-Fi interruption.
- **Solution**: Stop Section 8 by clicking the Stop button (or **Runtime > Interrupt execution**), wait 5 seconds, and re-run Section 8 to generate a fresh link.

### 5. Authentication Failure
- **Cause**: Incorrect username or password entered.
- **Solution**: Check the console output of Section 8 for the generated one-time credentials, or configure `PHARMAGUARD_AUTH_USER` and `PHARMAGUARD_AUTH_PASS` in Section 5.

### 6. Model Weight Download Error
- **Cause**: Intermittent network issue downloading `yolov8n.pt` from GitHub.
- **Solution**: Re-run Section 6 to re-trigger download.


---

## 11. Shutdown and Cleanup

When your surveillance demonstration is finished:

1. **Stop the Application**: Click the **Interrupt** (Stop) button on the code cell running Section 8, or press `Ctrl + M I` (or `Cmd + M I` on macOS).
2. **Release Colab GPU Resources**: To conserve your Google Colab compute units, disconnect the runtime from the top menu: **Runtime > Disconnect and delete runtime**.
